# JSON file

- JSON（JavaScript Object Notation）是一種輕量級的數據交換格式，易於人讀寫，也易於機器解析和生成。
- 以純文字為基礎，用來儲存簡單的資料，幾乎所有與網路開發相關的語言都會有JSON函式庫, 適合用於數據交換
- JSON 主要數據結構 (data structures)
  - Data (資料): 鍵值對 “name” : “John”
  - Object (物件): 由 {} 包圍，包含多個Data (鍵值對) ，例如：{"name": "Alice", "age": 25}
  - Array (陣列): 由 [] 包圍，包含多個Object或其他數據類型，<br>
    - employees包含3個Objects, each object is a record of a person
    - courses 包含兩個字串
- JSON 支持以下數據類型 (data types)
  - 字符串(String): 必須用雙引號括起來，例如 "Hello"
  - 數字(Number):不需要引號，例如 42 或 3.14
  - 布林值(Boolean): true 或 false (lower case)
  - 空值(Null): 用 null 表示
```JSON
{
"employees":[
  {"firstName":"John", "lastName":"Doe"},
  {"firstName":"Anna", "lastName":"Smith"},
  {"firstName":"Peter", "lastName":"Jones"}
]
}
```

```JSON
{
  "name": "Alice",
  "age": 25,
  "isStudent": false,
  "courses": ["Math", "Science"],
  "address": {
    "street": "123 Main St",
    "city": "Wonderland"
  }
}
```


# JSON Validation (https://jsonlint.com)

[Wrong JSON](/Users/jacky/Documents/交大教學/Python講義4/json/wrong_sample.json)
```JSON
{
    "name": 'Frieda',
    "address": {
        "work": null, //Doesn't pay rent either
        "home": "Berlin",
    },
    "friends": [
        {
            "name": "Philipp",
            "hobbies": ["eating", "sleeping", "reading",]
        }
    ]
}
```

# Read JSON file

[Score.json](/Users/jacky/Lecture-Python/files/json/score.json)

|id|name|chinese|math|english
|--|----|-------|----|-------
|1|John|90|80|70
|2|Mary|55|60|75
|3|Tom|90|95|100



In [2]:
# Import json module to read score.json file and print student id and name
from pathlib import Path
import json
file_path = Path.cwd() / ".." / 'json' / 'score.json'
with open(file_path, 'r', encoding='utf-8') as f:
  contents = json.load(f)  # 讀取json檔案, 讀回來一個list, element 是 dictionary
  for content in contents:
      student_id = content['id']
      name = content['name']
      print(f'{student_id=}, {name=}')

student_id=1, name='John'
student_id=2, name='Mary'
student_id=3, name='Tom'


In [3]:
# Import pandas module to read score.json file and print student id and name
from pathlib import Path
import pandas as pd
file_path = Path.cwd() / ".." / 'json' / 'score.json'
df = pd.read_json(file_path)
df.head()

,id,name,chinese,math,english
0,1,John,90,80,NaN
1,2,Mary,55,60,75.0
2,3,Tom,90,95,100.0


# Case study

## 112年新竹市政府公務人員考試新進人員人數
[政府資料開放平台](https://opendata.hccg.gov.tw/OpenDataDetail.aspx?n=1&s=1513)
```json
[
  {
    "序號": "1",
    "考試種類及等別": "公務人員高考三級",
    "新竹市政府公務人員考試新進人員人數": "7"
  },
  {
    "序號": "2",
    "考試種類及等別": "公務人員普通考試",
    "新竹市政府公務人員考試新進人員人數": "5"
  },
  {
    "序號": "3",
    "考試種類及等別": "地方政府特考四等",
    "新竹市政府公務人員考試新進人員人數": "1"
  }
]
```

In [4]:
import requests
# import json
import pandas as pd
from io import StringIO
url = 'https://odws.hccg.gov.tw/001/Upload/25/opendataback/9059/1513/bf9c3fd0-c5c5-4199-816f-089161e6e1b4.json'
response = requests.get(url)
if response.status_code != 200:
    print('Fail to open')

# df = pd.read_json(StringIO(response.text))  # 將字串轉成json物件, json物件是list, element是dict
# df.head()

df = pd.DataFrame(response.json())
df.head()


,序號,考試種類及等別,新竹市政府公務人員考試新進人員人數
0,1,公務人員高考三級,7
1,2,公務人員普通考試,5
2,3,地方政府特考四等,1


## 下載政府公開資料 JSON Format

In [5]:
import requests
from requests.exceptions import JSONDecodeError
import json
import pandas as pd

url = 'https://stats.moe.gov.tw/files/detail/112/112_base2.json'
response = requests.get(url)

# 檢查HTTP回應碼是否為200(requests.code.ok)
if response.status_code != 200:
    print('Fail to open')
    
# 🔧 Fix UTF-8 BOM issue
try:
    # Method 1: Try direct response.json() first
    df = pd.DataFrame(response.json())
    print("✅ Method 1 (response.json()) worked!")
except JSONDecodeError as e:
    print(f"❌ Method 1 failed: {e}")
    
    # Method 2: Remove BOM and use json.loads()
    try:
        # Remove UTF-8 BOM if present
        text_content = response.text
        if text_content.startswith('\ufeff'):
            text_content = text_content[1:]  # Remove BOM
        
        data = json.loads(text_content)
        df = pd.DataFrame(data)
        print("✅ Method 2 (BOM removal + json.loads()) worked!")
    except Exception as e2:
        print(f"❌ Method 2 also failed: {e2}")
        
        # Method 3: Use response.content with proper encoding
        try:
            # Decode content with utf-8-sig to handle BOM automatically
            content_decoded = response.content.decode('utf-8-sig')
            data = json.loads(content_decoded)
            df = pd.DataFrame(data)
            print("✅ Method 3 (utf-8-sig decoding) worked!")
        except Exception as e3:
            print(f"❌ All methods failed: {e3}")

# Display the result if successful
if 'df' in locals():
    print(f"\n📊 DataFrame shape: {df.shape}")
    print("First few rows:")
    print(df.head())

❌ Method 1 failed: Unexpected UTF-8 BOM (decode using utf-8-sig): line 1 column 1 (char 0)
✅ Method 2 (BOM removal + json.loads()) worked!

📊 DataFrame shape: (3449, 29)
First few rows:
   學年度 縣市代碼 縣市名稱    學校代碼        學校名稱 等級別   等級名稱 日夜別 日夜別名稱 群別代碼  ... 二年級男學生數  \
0  112   01  新北市  010301  國立華僑高級中等學校   A    普通科   D   日間部   11  ...      13   
1  112   01  新北市  010301  國立華僑高級中等學校   J  附設國中部   D   日間部       ...       8   
2  112   01  新北市  010301  國立華僑高級中等學校   O   綜合高中   D   日間部   11  ...       0   
3  112   01  新北市  010301  國立華僑高級中等學校   O   綜合高中   D   日間部   11  ...      68   
4  112   01  新北市  010301  國立華僑高級中等學校   O   綜合高中   D   日間部   11  ...     134   

  二年級女學生數 三年級男學生數  三年級女學生數  四年級男學生數  四年級女學生數  延修生男學生數  延修生女學生數  上學年畢業生數男  \
0      11      17        7        0        0        0        0        13   
1       6       5        9        0        0        0        0         6   
2       0       0        0        0        0        0        0         0   
3     111      74      123        0

# W3 School

- [JSON] https://www.w3schools.com/python/exercise.asp?x=xrcise_json1